# 🎨 Style Transfer Video Processing

This notebook allows you to apply neural style transfer to videos using various pre-trained models.

**Features:**
- Multiple style models (mosaic, candy, rain princess, etc.)
- GPU acceleration
- Audio preservation
- Easy upload/download interface

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install opencv-python moviepy pillow tqdm click colorama

# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

In [ ]:
# Clone the stylize-video repository
!git clone https://github.com/your-username/stylize-video.git
%cd stylize-video

# Verify installation
!python -m stylize_video list-models

In [ ]:
# Upload your input video
from google.colab import files
import os

print('📁 Please upload your input video:')
uploaded = files.upload()

# Get the uploaded filename
input_filename = list(uploaded.keys())[0]
print(f'✅ Uploaded: {input_filename}')
print(f'📊 File size: {len(uploaded[input_filename]) / (1024*1024):.1f} MB')

In [ ]:
# Configuration
INPUT_VIDEO = input_filename
OUTPUT_VIDEO = f"styled_{input_filename}"
STYLE_MODEL = "mosaic"  # Options: mosaic, candy, rain_princess, udnie
FPS = 30
RESIZE = 512
KEEP_AUDIO = True

print("🎬 Processing Configuration:")
print(f"Input: {INPUT_VIDEO}")
print(f"Output: {OUTPUT_VIDEO}")
print(f"Style: {STYLE_MODEL}")
print(f"FPS: {FPS}")
print(f"Resize: {RESIZE}px")
print(f"Keep Audio: {KEEP_AUDIO}")

In [ ]:
# Run style transfer processing
import time

start_time = time.time()

!python -m stylize_video stylize \
  --input "{INPUT_VIDEO}" \
  --output "{OUTPUT_VIDEO}" \
  --model {STYLE_MODEL} \
  --backend local \
  --fps {FPS} \
  --resize {RESIZE} \
  {"--keep-audio" if KEEP_AUDIO else "--no-audio"} \
  --verbose

elapsed = time.time() - start_time
print(f"\n✅ Processing completed in {elapsed:.1f} seconds!")

In [ ]:
# Preview the result (first few frames)
import cv2
import matplotlib.pyplot as plt
from IPython.display import HTML

if os.path.exists(OUTPUT_VIDEO):
    # Display video info
    cap = cv2.VideoCapture(OUTPUT_VIDEO)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps
    
    print(f"📊 Output Video Info:")
    print(f"Resolution: {width}x{height}")
    print(f"FPS: {fps:.2f}")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Frames: {frame_count}")
    
    # Show first frame
    ret, frame = cap.read()
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 6))
        plt.imshow(frame_rgb)
        plt.title(f"First Frame - {STYLE_MODEL.title()} Style")
        plt.axis('off')
        plt.show()
    
    cap.release()
else:
    print("❌ Output video not found. Check the processing cell for errors.")

In [ ]:
# Download the stylized video
from google.colab import files
import os

if os.path.exists(OUTPUT_VIDEO):
    file_size = os.path.getsize(OUTPUT_VIDEO) / (1024*1024)
    print(f"📁 Downloading {OUTPUT_VIDEO} ({file_size:.1f} MB)...")
    files.download(OUTPUT_VIDEO)
    print('✅ Download started! Check your browser\'s download folder.')
else:
    print('❌ Output video not found. Make sure the processing completed successfully.')

In [ ]:
# Clean up temporary files (optional)
import shutil

cleanup = input("Clean up temporary files? (y/N): ")
if cleanup.lower() == 'y':
    # Remove temp directories
    for temp_dir in ['temp', 'models/checkpoints']:
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
            print(f"🧹 Cleaned: {temp_dir}")
    
    print("✅ Cleanup completed!")
else:
    print("Skipping cleanup")